# NeoNatal Watch AI — Phase 2: Data Preprocessing

> **DEMO / SYNTHETIC DATA — NOT FOR CLINICAL USE**
>
> This notebook demonstrates the preprocessing pipeline applied to the synthetic dataset.

---

## Preprocessing Steps:
1. **Outlier Detection**: Identify physiologically impossible values.
2. **Missing Value Handling**: Forward-fill and patient-specific interpolation.
3. **Resampling**: Standardize high-frequency data to 1-minute intervals.
4. **Patient-Level Split**: Avoid data leakage by keeping patients entirely in one split (Train/Val/Test).
5. **Normalization**: Min-Max scaling fit *only* on training data.
6. **Windowing**: Create 3D time-series windows (e.g., 30 timesteps) for deep learning models.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from ml.data.data_loader import load_synthetic_data
from ml.preprocessing.preprocess import run_preprocessing_pipeline

print('Imports OK')

In [ ]:
# Load raw synthetic data
df_raw = load_synthetic_data('../data/synthetic')
print(f"Raw Data Shape: {df_raw.shape}")

In [ ]:
# Run Pipeline
results = run_preprocessing_pipeline(
    df=df_raw,
    source="synthetic",
    target_seconds=60,
    scaler_method="minmax",
    train_frac=0.70,
    val_frac=0.15,
    test_frac=0.15,
    seed=42,
    output_dir=None, # Don't save files again in notebook
)


In [ ]:
print("=== Pipeline Results ===")
print(f"Train patients: {results['train_df']['patient_id'].nunique()}")
print(f"Val patients:   {results['val_df']['patient_id'].nunique()}")
print(f"Test patients:  {results['test_df']['patient_id'].nunique()}")
print()
print("=== Window Shapes ===")
print(f"X_train: {results['X_train'].shape}")
print(f"X_val:   {results['X_val'].shape}")
print(f"X_test:  {results['X_test'].shape}")

### Key Takeaways
- **Data Leakage Prevented**: Scaling parameters were learned strictly from `train_df`.
- **3D Tensors Ready**: The data was transformed into `(samples, timesteps, features)` shapes required for LSTMs and Transformers.